# Concurrency
Concurrency is the ability to run multiple tasks or parts of a program seemingly at the same time or overlapping in time. In Python, choosing the right concurrency model depends entirely on whether your workload is I/O-bound (waiting on network requests, file reads, or databases) or CPU-bound (heavy math, image processing, or data crunching).

## 1. The GIL (Global Interpreter Lock)
Before exploring Python's concurrency modules, you must understand the GIL.

What it is: The GIL is a mutex (lock) used by CPython (the default Python implementation) that prevents multiple native threads from executing Python bytecodes at once.

Why it exists: It protects memory management and reference counting from race conditions, making single-threaded programs fast and easy to integrate with C libraries.

The Impact: Because of the GIL, threads cannot run Python code in parallel on multiple CPU cores. For CPU-bound tasks, multithreading actually slows things down due to context-switching overhead. However, for I/O-bound tasks, threads work wonderfully because the GIL is released while waiting for external responses.

## 2. Threading (threading)
The threading module uses operating system-level threads. It is ideal for I/O-bound tasks because while one thread is waiting for a web response, another thread can execute.

In [ ]:
import threading
import time

def download_file(file_name):
    print(f"Starting download: {file_name}")
    time.sleep(2)  # Simulate network I/O wait
    print(f"Finished download: {file_name}")

# Create threads
t1 = threading.Thread(target=download_file, args=("file_1.zip",))
t2 = threading.Thread(target=download_file, args=("file_2.zip",))

# Start threads
t1.start()
t2.start()

# Wait for both threads to finish
t1.join()
t2.join()

print("All downloads complete!")

## 3. Multiprocessing (multiprocessing)
To bypass the GIL and leverage multiple CPU cores for CPU-bound tasks, Python provides the multiprocessing module. Instead of threads, it spawns separate, independent subprocesses, each with its own Python interpreter and memory space.

In [ ]:
import multiprocessing
import time

def heavy_calculation(number):
    print(f"Calculating square for {number}...")
    time.sleep(1)
    return number * number

if __name__ == "__main__":
    # Create a pool of processes matching CPU cores
    with multiprocessing.Pool(processes=2) as pool:
        results = pool.map(heavy_calculation, [10, 20])
    
    print(f"Results: {results}")

## 4. Asyncio, async, and await
asyncio is Python's built-in library for writing asynchronous code using the async and await syntax. Unlike threading, which relies on preemptive multitasking managed by the OS, asyncio uses cooperative multitasking. Tasks explicitly yield control when they have to wait, allowing other tasks to run on a single thread.

async def: Defines a coroutine function.

await: Pauses the coroutine until the awaited result is ready, letting other tasks execute in the meantime.

In [ ]:
import asyncio

async def fetch_data(api_name, delay):
    print(f"Fetching from {api_name}...")
    await asyncio.sleep(delay)  # Non-blocking pause
    print(f"Received data from {api_name}!")
    return {"api": api_name, "data": 42}

async def main():
    # Run multiple coroutines concurrently
    results = await asyncio.gather(
        fetch_data("API_1", 2),
        fetch_data("API_2", 1)
    )
    print(results)

# Run the async program
asyncio.run(main())

## 5. The Event Loop
The Event Loop is the core engine of asyncio. It tracks all asynchronous tasks, executes them, pauses them when they hit an await statement, switches to other ready tasks, and resumes them once their awaited resource becomes available.

You rarely manage the event loop manually anymore because asyncio.run() handles creation, execution, and cleanup automatically, but understanding its underlying loop mechanics is crucial for debugging.

## 6. Futures and Tasks
**Future:** A low-level object (asyncio.Future) that represents the eventual result of an asynchronous operation that isn't finished yet.

**Task:** A subclass of Future that wraps a coroutine. When you schedule a coroutine to run concurrently (e.g., using asyncio.create_task()), Python wraps it in a Task so the event loop can track its progress.

In [ ]:
async def sample_task():
    await asyncio.sleep(1)
    return "Done"

async def main():
    # Wrap coroutine in a Task to run it concurrently
    task = asyncio.create_task(sample_task())
    
    print(f"Is task done? {task.done()}")
    result = await task
    print(f"Task result: {result}")

asyncio.run(main())

## 7. Executors (ThreadPoolExecutor and ProcessPoolExecutor)
The concurrent.futures module provides a high-level, unified interface for running tasks asynchronously using pools of threads or processes without managing low-level thread lifecycle code manually.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def task(name):
    time.sleep(1)
    return f"Task {name} complete"

# Using a ThreadPoolExecutor as a context manager
with ThreadPoolExecutor(max_workers=3) as executor:
    # Submit tasks and get future objects back
    futures = [executor.submit(task, i) for i in range(3)]
    
    for f in futures:
        print(f.result())  # Retrieve results as they finish

### Best Practices & Common Pitfalls
***Match Concurrency to the Task Type:*** Use multiprocessing for CPU-heavy tasks (crunching numbers, video rendering). Use asyncio or threading for I/O-heavy tasks (web scraping, API calls, file downloading).

***Don't Mix Blocking Code with Asyncio:*** If you call a blocking function (like standard time.sleep() or synchronous requests like requests.get()) inside an asyncio loop, it freezes the entire single thread, ruining concurrency. Always use async-compatible equivalents (like asyncio.sleep() or httpx).

***Beware of Shared State Race Conditions:*** When using threading, multiple threads modifying the same variable simultaneously can cause data corruption. Use threading.Lock to synchronize access to shared data.